# Séance 6 · Exercices — Collecter des données avec une API · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐ Intermédiaire** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler :
1. Lis l'énoncé, puis code dans la cellule « À toi » (les variables à remplir sont déjà nommées).
2. Lance la cellule de vérification : ✅ c'est bon, ❌ on réessaie (l'indice est là pour ça).
3. Ouvre la solution **seulement après avoir essayé**.

Ce notebook tourne dans **Google Colab** : rien à installer. Pour exécuter une cellule : clique dedans puis `Maj + Entrée`.

Les gestes de la leçon, avec d'autres API gratuites et sans clé : les jours fériés (Nager.Date), une bibliothèque en ligne (Open Library), une fausse liste de tâches pour s'entraîner (JSONPlaceholder), et la recherche de villes et la météo d'Open-Meteo. Chaque appel a son plan B : sans réseau, le notebook va au bout avec des données de secours (relevées le 17/10/2026), et les vérifications marchent dans les deux cas.

## Préparation

`requests` et pandas, la fonction `demander(url, secours, params)` de la leçon, et les données de secours de chaque API.

In [ ]:
import io
import json
import time
import contextlib
import requests
import pandas as pd

def demander(url, secours, params=None):
    """Appelle l'API et renvoie son JSON ; si ça rate (pas de réseau, erreur), renvoie les données de secours."""
    try:
        reponse = requests.get(url, params=params, timeout=10)
        reponse.raise_for_status()
        return reponse.json()
    except Exception as erreur:
        print("Pas de réponse de l'API, on utilise les données de secours :", erreur)
        return secours

def statut(url):
    """Le code de réponse d'une URL (200, 404...), ou None s'il n'y a pas de réseau."""
    try:
        return requests.get(url, timeout=10).status_code
    except requests.RequestException:
        return None

def capturer(fonction, *args, **kwargs):
    """Lance la fonction et renvoie (son résultat, ce qu'elle a affiché) : pour les vérifications."""
    tampon = io.StringIO()
    with contextlib.redirect_stdout(tampon):
        resultat = fonction(*args, **kwargs)
    return resultat, tampon.getvalue()

# Les données de secours (réponses réelles des API, relevées le 17/10/2026)
SECOURS_FERIES = {
    "FR": [{'date': '2026-01-01', 'localName': "Jour de l'an", 'name': "New Year's Day", 'global': True},
 {'date': '2026-04-06', 'localName': 'Lundi de Pâques', 'name': 'Easter Monday', 'global': True},
 {'date': '2026-05-01', 'localName': 'Fête du Travail', 'name': 'Labour Day', 'global': True},
 {'date': '2026-05-08', 'localName': 'Victoire 1945', 'name': 'Victory in Europe Day', 'global': True},
 {'date': '2026-05-14', 'localName': 'Ascension', 'name': 'Ascension Day', 'global': True},
 {'date': '2026-05-25', 'localName': 'Lundi de Pentecôte', 'name': 'Whit Monday', 'global': True},
 {'date': '2026-07-14', 'localName': 'Fête nationale', 'name': 'Bastille Day', 'global': True},
 {'date': '2026-08-15', 'localName': 'Assomption', 'name': 'Assumption Day', 'global': True},
 {'date': '2026-11-01', 'localName': 'Toussaint', 'name': "All Saints' Day", 'global': True},
 {'date': '2026-11-11', 'localName': 'Armistice 1918', 'name': 'Armistice Day', 'global': True},
 {'date': '2026-12-25', 'localName': 'Noël', 'name': 'Christmas Day', 'global': True}],
    "DE": [{'date': '2026-01-01', 'localName': 'Neujahr', 'name': "New Year's Day", 'global': True},
 {'date': '2026-01-06', 'localName': 'Heilige Drei Könige', 'name': 'Epiphany', 'global': False},
 {'date': '2026-03-08', 'localName': 'Internationaler Frauentag', 'name': "International Women's Day", 'global': False},
 {'date': '2026-04-03', 'localName': 'Karfreitag', 'name': 'Good Friday', 'global': True},
 {'date': '2026-04-05', 'localName': 'Ostersonntag', 'name': 'Easter Sunday', 'global': False},
 {'date': '2026-04-06', 'localName': 'Ostermontag', 'name': 'Easter Monday', 'global': True},
 {'date': '2026-05-01', 'localName': 'Tag der Arbeit', 'name': 'Labour Day', 'global': True},
 {'date': '2026-05-14', 'localName': 'Christi Himmelfahrt', 'name': 'Ascension Day', 'global': True},
 {'date': '2026-05-24', 'localName': 'Pfingstsonntag', 'name': 'Pentecost', 'global': False},
 {'date': '2026-05-25', 'localName': 'Pfingstmontag', 'name': 'Whit Monday', 'global': True},
 {'date': '2026-06-04', 'localName': 'Fronleichnam', 'name': 'Corpus Christi', 'global': False},
 {'date': '2026-08-15', 'localName': 'Mariä Himmelfahrt', 'name': 'Assumption Day', 'global': False},
 {'date': '2026-09-20', 'localName': 'Weltkindertag', 'name': "World Children's Day", 'global': False},
 {'date': '2026-10-03', 'localName': 'Tag der Deutschen Einheit', 'name': 'German Unity Day', 'global': True},
 {'date': '2026-10-31', 'localName': 'Reformationstag', 'name': 'Reformation Day', 'global': False},
 {'date': '2026-11-01', 'localName': 'Allerheiligen', 'name': "All Saints' Day", 'global': False},
 {'date': '2026-11-18', 'localName': 'Buß- und Bettag', 'name': 'Repentance and Prayer Day', 'global': False},
 {'date': '2026-12-25', 'localName': 'Erster Weihnachtstag', 'name': 'Christmas Day', 'global': True},
 {'date': '2026-12-26', 'localName': 'Zweiter Weihnachtstag', 'name': "St. Stephen's Day", 'global': True}],
}
SECOURS_TACHES = [{'userId': 3,
  'id': 41,
  'title': 'aliquid amet impedit consequatur aspernatur placeat eaque fugiat suscipit',
  'completed': False},
 {'userId': 3, 'id': 42, 'title': 'rerum perferendis error quia ut eveniet', 'completed': False},
 {'userId': 3, 'id': 43, 'title': 'tempore ut sint quis recusandae', 'completed': True},
 {'userId': 3, 'id': 44, 'title': 'cum debitis quis accusamus doloremque ipsa natus sapiente omnis', 'completed': True},
 {'userId': 3, 'id': 45, 'title': 'velit soluta adipisci molestias reiciendis harum', 'completed': False},
 {'userId': 3, 'id': 46, 'title': 'vel voluptatem repellat nihil placeat corporis', 'completed': False},
 {'userId': 3, 'id': 47, 'title': 'nam qui rerum fugiat accusamus', 'completed': False},
 {'userId': 3, 'id': 48, 'title': 'sit reprehenderit omnis quia', 'completed': False},
 {'userId': 3, 'id': 49, 'title': 'ut necessitatibus aut maiores debitis officia blanditiis velit et', 'completed': False},
 {'userId': 3, 'id': 50, 'title': 'cupiditate necessitatibus ullam aut quis dolor voluptate', 'completed': True},
 {'userId': 3, 'id': 51, 'title': 'distinctio exercitationem ab doloribus', 'completed': False},
 {'userId': 3, 'id': 52, 'title': 'nesciunt dolorum quis recusandae ad pariatur ratione', 'completed': False},
 {'userId': 3, 'id': 53, 'title': 'qui labore est occaecati recusandae aliquid quam', 'completed': False},
 {'userId': 3, 'id': 54, 'title': 'quis et est ut voluptate quam dolor', 'completed': True},
 {'userId': 3, 'id': 55, 'title': 'voluptatum omnis minima qui occaecati provident nulla voluptatem ratione', 'completed': True},
 {'userId': 3, 'id': 56, 'title': 'deleniti ea temporibus enim', 'completed': True},
 {'userId': 3, 'id': 57, 'title': 'pariatur et magnam ea doloribus similique voluptatem rerum quia', 'completed': False},
 {'userId': 3, 'id': 58, 'title': 'est dicta totam qui explicabo doloribus qui dignissimos', 'completed': False},
 {'userId': 3, 'id': 59, 'title': 'perspiciatis velit id laborum placeat iusto et aliquam odio', 'completed': False},
 {'userId': 3, 'id': 60, 'title': 'et sequi qui architecto ut adipisci', 'completed': True}]
SECOURS_LIVRES = {'numFound': 517,
 'docs': [{'author_name': ['Antoine de Saint-Exupéry'], 'first_publish_year': 1943, 'title': 'Le petit prince'},
          {'author_name': ['Antoine de Saint-Exupéry'], 'first_publish_year': 1990, 'title': 'El Principito'},
          {'author_name': ['Antoine de Saint-Exupéry', 'Gallimard'], 'first_publish_year': 1999, 'title': 'Le Petit Prince'},
          {'author_name': ['Antoine de Saint-Exupéry'], 'first_publish_year': 2002, 'title': 'Le petit prince'},
          {'author_name': ['Antoine de Saint-Exupéry'], 'title': 'Hoàng tu bé'}]}
SECOURS_VILLES = {'results': [{'name': 'Brest', 'country_code': 'BY', 'country': 'Biélorussie', 'latitude': 52.10894, 'longitude': 23.71749},
             {'name': 'Brest', 'country_code': 'FR', 'country': 'France', 'latitude': 48.39029, 'longitude': -4.48628},
             {'name': 'Brest', 'country_code': 'DE', 'country': 'Allemagne', 'latitude': 53.45018, 'longitude': 9.38401},
             {'name': 'Brest', 'country_code': 'BG', 'country': 'Bulgarie', 'latitude': 43.6427, 'longitude': 24.59162},
             {'name': 'Brest', 'country_code': 'BG', 'country': 'Bulgarie', 'latitude': 42.46233, 'longitude': 22.60447}]}
SECOURS_SEMAINE = {'daily': {'time': ['2026-10-17', '2026-10-18', '2026-10-19', '2026-10-20', '2026-10-21', '2026-10-22', '2026-10-23'],
           'temperature_2m_max': [15.5, 14.9, 13.6, 14.0, 13.2, 15.4, 14.7],
           'precipitation_sum': [0.0, 0.3, 1.3, 1.4, 1.15, 10.2, 4.5]}}

resultats = {}

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = un booléen, ou une fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else " — pas encore, relis l'énoncé ou ouvre l'indice."))

print("Prêt !")

## Exercice 1 ⭐ · Le code de réponse

JSONPlaceholder est une fausse API faite pour s'entraîner. Avec la fonction `statut(url)`, récupère le code de réponse de deux adresses :

- `code_ok` : `https://jsonplaceholder.typicode.com/todos/1` (la tâche numéro 1 existe),
- `code_absent` : `https://jsonplaceholder.typicode.com/todos/9999` (elle n'existe pas).

Résultat attendu : `200` et `404` (ou `None` deux fois si tu n'as pas de réseau).

<details><summary>Indice</summary>

`code_ok = statut("https://jsonplaceholder.typicode.com/todos/1")`, pareil pour l'autre.

</details>

In [ ]:
# À toi
code_ok = None
code_absent = None

print(code_ok, code_absent)

In [ ]:
verifier("Exercice 1", code_ok in (200, None) and (code_ok, code_absent) == (statut("https://jsonplaceholder.typicode.com/todos/1"), statut("https://jsonplaceholder.typicode.com/todos/9999")))

<details><summary>Solution</summary>

```python
code_ok = statut("https://jsonplaceholder.typicode.com/todos/1")
code_absent = statut("https://jsonplaceholder.typicode.com/todos/9999")

print(code_ok, code_absent)
```

Un code 404 n'est pas un plantage : le site a bien répondu, pour dire « ça n'existe pas ». C'est à toi de vérifier le code avant de lire la réponse.

</details>

## Exercice 2 ⭐ · Lire un JSON

Voici, écrite à la main, une réponse JSON du genre de celles d'Open Library. `json.loads` transforme le texte en dictionnaire.
Remplis :

- `titre` : le titre du livre,
- `annee` : l'année de première publication,
- `nb_auteurs` : combien d'auteurs sont listés.

<details><summary>Indice</summary>

`livre = json.loads(texte)`, puis `livre["title"]`, `livre["first_publish_year"]` et `len(livre["author_name"])`.

</details>

In [ ]:
# À toi
texte = '{"title": "Vingt mille lieues sous les mers", "author_name": ["Jules Verne", "Alphonse de Neuville"], "first_publish_year": 1870}'

titre = None
annee = None
nb_auteurs = None

print(titre, annee, nb_auteurs)

In [ ]:
verifier("Exercice 2", titre == "Vingt mille lieues sous les mers" and annee == 1870 and nb_auteurs == 2)

<details><summary>Solution</summary>

```python
texte = '{"title": "Vingt mille lieues sous les mers", "author_name": ["Jules Verne", "Alphonse de Neuville"], "first_publish_year": 1870}'
livre = json.loads(texte)

titre = livre["title"]
annee = livre["first_publish_year"]
nb_auteurs = len(livre["author_name"])

print(titre, annee, nb_auteurs)
```

Alphonse de Neuville est l'illustrateur : une API rend ce qu'elle a dans sa base, à toi de savoir ce que veut dire chaque champ.

</details>

## Exercice 3 ⭐ · Compter dans une liste de dictionnaires

La liste des tâches de l'utilisateur numéro 3 : `https://jsonplaceholder.typicode.com/todos?userId=3`. Chaque tâche est un dictionnaire
avec une clé `completed` (`True` si elle est faite).

- `taches` : la réponse (avec `demander`, les paramètres `{"userId": 3}` et le secours `SECOURS_TACHES`),
- `nb_taches` : le nombre de tâches,
- `nb_finies` : combien sont faites.

<details><summary>Indice</summary>

`sum(...)` compte les `True` : `sum(t["completed"] for t in taches)`.

</details>

In [ ]:
# À toi
taches = None
nb_taches = None
nb_finies = None

print(nb_taches, "tâches,", nb_finies, "finies")

In [ ]:
verifier("Exercice 3", isinstance(taches, list) and nb_taches == len(taches) and nb_finies == sum(t["completed"] for t in taches) and all(t["userId"] == 3 for t in taches))

<details><summary>Solution</summary>

```python
taches = demander("https://jsonplaceholder.typicode.com/todos", SECOURS_TACHES, params={"userId": 3})
nb_taches = len(taches)
nb_finies = sum(t["completed"] for t in taches)

print(nb_taches, "tâches,", nb_finies, "finies")
```

</details>

## Exercice 4 ⭐⭐ · Les jours fériés, en tableau

L'API Nager.Date donne les jours fériés d'un pays pour une année : `https://date.nager.at/api/v3/PublicHolidays/2026/FR`.

- `feries` : un tableau pandas construit à partir de la réponse (une ligne par jour férié),
- `nb_feries` : le nombre de jours fériés,
- `premier` : le nom local (`localName`) du premier de l'année.

<details><summary>Indice</summary>

La réponse est une liste de dictionnaires : `pd.DataFrame(reponse)` en fait directement un tableau. Le secours : `SECOURS_FERIES["FR"]`.

</details>

In [ ]:
# À toi
feries = None
nb_feries = None
premier = None

print(nb_feries, "jours fériés, le premier :", premier)

In [ ]:
verifier("Exercice 4", isinstance(feries, pd.DataFrame) and nb_feries == len(feries) and nb_feries >= 10 and premier == feries.sort_values("date").iloc[0]["localName"])

<details><summary>Solution</summary>

```python
reponse = demander("https://date.nager.at/api/v3/PublicHolidays/2026/FR", SECOURS_FERIES["FR"])
feries = pd.DataFrame(reponse)
nb_feries = len(feries)
premier = feries.sort_values("date").iloc[0]["localName"]

print(nb_feries, "jours fériés, le premier :", premier)
feries[["date", "localName"]]
```

</details>

## Exercice 5 ⭐⭐ · Des paramètres : chercher un livre

Open Library cherche dans des millions de livres : `https://openlibrary.org/search.json`, avec les paramètres
`q` (la recherche), `limit` (combien de résultats) et `fields` (les champs voulus).

Cherche `"le petit prince"`, 5 résultats, champs `"title,author_name,first_publish_year"` :

- `reponse_livres` : la réponse complète (secours : `SECOURS_LIVRES`),
- `nb_trouves` : le nombre total de livres trouvés (clé `numFound`),
- `titres` : la liste des titres des 5 résultats (dans la clé `docs`).

<details><summary>Indice</summary>

Les paramètres dans un dictionnaire : `params={"q": "le petit prince", "limit": 5, "fields": "title,author_name,first_publish_year"}`. Puis `[d["title"] for d in reponse_livres["docs"]]`.

</details>

In [ ]:
# À toi
reponse_livres = None
nb_trouves = None
titres = None

print(nb_trouves, "livres trouvés")
print(titres)

In [ ]:
verifier("Exercice 5", isinstance(reponse_livres, dict) and nb_trouves == reponse_livres["numFound"] and titres == [d["title"] for d in reponse_livres["docs"]] and len(titres) <= 5)

<details><summary>Solution</summary>

```python
params = {"q": "le petit prince", "limit": 5, "fields": "title,author_name,first_publish_year"}
reponse_livres = demander("https://openlibrary.org/search.json", SECOURS_LIVRES, params=params)
nb_trouves = reponse_livres["numFound"]
titres = [d["title"] for d in reponse_livres["docs"]]

print(nb_trouves, "livres trouvés")
print(titres)
```

Le même livre revient plusieurs fois (traductions, éditions) : une API de bibliothèque compte des éditions, pas des œuvres.

</details>

## Exercice 6 ⭐⭐ · Une valeur qui manque

Dans la réponse de l'exercice 5 (redemandée dans la cellule), certains livres n'ont pas d'année. `d["first_publish_year"]` planterait sur eux ;
`d.get("first_publish_year")` renvoie `None` à la place.

- `livres` : un tableau avec deux colonnes, `titre` et `annee`,
- `nb_sans_annee` : le nombre de livres sans année,
- `plus_ancien` : le titre du livre à l'année la plus ancienne.

<details><summary>Indice</summary>

Une liste de dictionnaires `{"titre": d["title"], "annee": d.get("first_publish_year")}`, puis `pd.DataFrame`. Les `None` deviennent des cases vides : `.isna().sum()`, et `sort_values("annee")` les met à la fin.

</details>

In [ ]:
# À toi
params = {"q": "le petit prince", "limit": 5, "fields": "title,author_name,first_publish_year"}
reponse_livres = demander("https://openlibrary.org/search.json", SECOURS_LIVRES, params=params)

livres = None
nb_sans_annee = None
plus_ancien = None

print(livres)

In [ ]:
verifier("Exercice 6", isinstance(livres, pd.DataFrame) and list(livres.columns) == ["titre", "annee"] and nb_sans_annee == livres["annee"].isna().sum() and plus_ancien == livres.sort_values("annee").iloc[0]["titre"])

<details><summary>Solution</summary>

```python
params = {"q": "le petit prince", "limit": 5, "fields": "title,author_name,first_publish_year"}
reponse_livres = demander("https://openlibrary.org/search.json", SECOURS_LIVRES, params=params)

livres = pd.DataFrame([{"titre": d["title"], "annee": d.get("first_publish_year")} for d in reponse_livres["docs"]])
nb_sans_annee = livres["annee"].isna().sum()
plus_ancien = livres.sort_values("annee").iloc[0]["titre"]

print(livres)
```

`.get(clé)` est le réflexe avec une API : un champ peut manquer sur une fiche et pas sur une autre, sans prévenir.

</details>

## Exercice 7 ⭐⭐ · Insister avant d'abandonner

Une API gratuite rate parfois un appel, puis répond au suivant. Écris `demander_insistant(url, secours, essais=3, params=None)` qui :

- essaie l'appel jusqu'à `essais` fois ;
- à chaque échec, affiche `Essai 1 raté`, `Essai 2 raté`… et attend une seconde avant de réessayer ;
- renvoie le JSON dès qu'un essai réussit, ou le secours si tous ont raté.

La vérification appelle une adresse qui n'existe pas et compte les messages affichés.

<details><summary>Indice</summary>

Une boucle `for numero in range(1, essais + 1):`, avec dedans le `try` de `demander` : un `return` dans le `try`, un `print(f"Essai {numero} raté")` et un `time.sleep(1)` dans le `except`. Après la boucle : `return secours`.

</details>

In [ ]:
# À toi
def demander_insistant(url, secours, essais=3, params=None):
    return secours

print(demander_insistant("https://jsonplaceholder.typicode.com/todos/1", {"title": "secours"})["title"])

In [ ]:
verifier("Exercice 7", (lambda rt: rt[0] == {"rien": True} and rt[1].count("raté") == 2)(capturer(demander_insistant, "https://jsonplaceholder.typicode.com/todos/9999", {"rien": True}, essais=2)))

<details><summary>Solution</summary>

```python
def demander_insistant(url, secours, essais=3, params=None):
    for numero in range(1, essais + 1):
        try:
            reponse = requests.get(url, params=params, timeout=10)
            reponse.raise_for_status()
            return reponse.json()
        except Exception:
            print(f"Essai {numero} raté")
            time.sleep(1)
    return secours

print(demander_insistant("https://jsonplaceholder.typicode.com/todos/1", {"title": "secours"})["title"])
```

Insister a une limite : trois essais espacés, c'est poli ; cent essais par seconde, c'est une attaque, et le site te bloquera.

</details>

## Exercice 8 ⭐⭐ · Le bon Brest

Open-Meteo a aussi une recherche de villes : `https://geocoding-api.open-meteo.com/v1/search`, avec les paramètres `name`, `count` (combien de résultats) et `language`.

Cherche `"Brest"`, 5 résultats, en français. Surprise : le premier n'est pas en France.

- `villes` : la liste des résultats (la clé `results` de la réponse ; secours : `SECOURS_VILLES`),
- `premier_pays` : le pays (`country`) du premier résultat,
- `brest_fr` : le premier résultat dont `country_code` vaut `"FR"`,
- `lat_brest`, `lon_brest` : sa latitude et sa longitude.

<details><summary>Indice</summary>

Pour trouver le premier qui convient : `[v for v in villes if v["country_code"] == "FR"][0]`.

</details>

In [ ]:
# À toi
villes = None
premier_pays = None
brest_fr = None
lat_brest = None
lon_brest = None

print(premier_pays, "·", lat_brest, lon_brest)

In [ ]:
verifier("Exercice 8", isinstance(villes, list) and premier_pays == villes[0]["country"] and brest_fr["country_code"] == "FR" and abs(lat_brest - 48.39) < 0.1 and abs(lon_brest + 4.49) < 0.1)

<details><summary>Solution</summary>

```python
reponse = demander("https://geocoding-api.open-meteo.com/v1/search", SECOURS_VILLES,
                   params={"name": "Brest", "count": 5, "language": "fr"})
villes = reponse["results"]
premier_pays = villes[0]["country"]
brest_fr = [v for v in villes if v["country_code"] == "FR"][0]
lat_brest, lon_brest = brest_fr["latitude"], brest_fr["longitude"]

print(premier_pays, "·", lat_brest, lon_brest)
```

L'API ne devine pas : elle classe les Brest par taille, et celui de Biélorussie est plus grand. Avec `count=1`, ton programme aurait donné la météo de Biélorussie à un Breton. On vérifie toujours ce qu'une API répond, surtout quand la réponse « a l'air » juste.

</details>

## Exercice 9 ⭐⭐⭐ · La météo de la semaine à Brest

Avec les coordonnées de Brest (48.39, -4.49), demande à Open-Meteo les prévisions **jour par jour** : paramètres `latitude`, `longitude`,
`daily="temperature_2m_max,precipitation_sum"` et `timezone="Europe/Paris"` (secours : `SECOURS_SEMAINE`).

- `semaine` : un tableau avec trois colonnes, `jour` (de vraies dates), `temperature_max` et `pluie`,
- `jour_le_plus_chaud` : la date (en texte, comme `"2026-10-17"`) du jour le plus chaud,
- `pluie_totale` : la pluie totale de la semaine, en mm, arrondie à 1 décimale.

<details><summary>Indice</summary>

Les trois listes sont sous la clé `daily` : `d["time"]`, `d["temperature_2m_max"]`, `d["precipitation_sum"]`. Pour la date du max : `semaine.sort_values("temperature_max").iloc[-1]["jour"]`, puis `.strftime("%Y-%m-%d")`.

</details>

In [ ]:
# À toi
params = {"latitude": 48.39, "longitude": -4.49, "daily": "temperature_2m_max,precipitation_sum", "timezone": "Europe/Paris"}

semaine = None
jour_le_plus_chaud = None
pluie_totale = None

print(semaine)

In [ ]:
verifier("Exercice 9", isinstance(semaine, pd.DataFrame) and list(semaine.columns) == ["jour", "temperature_max", "pluie"] and str(semaine["jour"].dtype).startswith("datetime") and jour_le_plus_chaud == semaine.sort_values("temperature_max").iloc[-1]["jour"].strftime("%Y-%m-%d") and pluie_totale == round(semaine["pluie"].sum(), 1))

<details><summary>Solution</summary>

```python
params = {"latitude": 48.39, "longitude": -4.49, "daily": "temperature_2m_max,precipitation_sum", "timezone": "Europe/Paris"}
d = demander("https://api.open-meteo.com/v1/forecast", SECOURS_SEMAINE, params=params)["daily"]

semaine = pd.DataFrame({"jour": pd.to_datetime(d["time"]), "temperature_max": d["temperature_2m_max"], "pluie": d["precipitation_sum"]})
jour_le_plus_chaud = semaine.sort_values("temperature_max").iloc[-1]["jour"].strftime("%Y-%m-%d")
pluie_totale = round(semaine["pluie"].sum(), 1)

print(semaine)
```

La vérification relit **tes** données : la météo change chaque jour, donc le bon résultat aussi.

</details>

## Exercice 10 ⭐⭐⭐ · Garder ce qu'on a collecté

Sauvegarde les jours fériés français de 2026 dans `feries_fr.csv`, puis relis le fichier dans `feries_relu`.
Le tableau est redemandé dans la cellule, pour que l'exercice marche seul.

Résultat attendu : `feries_relu` a les mêmes lignes et les mêmes colonnes que `feries`.

<details><summary>Indice</summary>

`feries.to_csv("feries_fr.csv", index=False)` (sans `index=False`, une colonne de numéros s'ajoute), puis `pd.read_csv("feries_fr.csv")`.

</details>

In [ ]:
# À toi
feries = pd.DataFrame(demander("https://date.nager.at/api/v3/PublicHolidays/2026/FR", SECOURS_FERIES["FR"]))

feries_relu = None

print(feries_relu.shape if feries_relu is not None else None, feries.shape)

In [ ]:
verifier("Exercice 10", feries_relu is not None and feries_relu.shape == feries.shape and list(feries_relu.columns) == list(feries.columns))

<details><summary>Solution</summary>

```python
feries = pd.DataFrame(demander("https://date.nager.at/api/v3/PublicHolidays/2026/FR", SECOURS_FERIES["FR"]))

feries.to_csv("feries_fr.csv", index=False)
feries_relu = pd.read_csv("feries_fr.csv")

print(feries_relu.shape if feries_relu is not None else None, feries.shape)
```

Les dates relues sont redevenues du texte : un CSV ne garde pas les types. On les reconvertit avec `pd.to_datetime` après chaque lecture.

</details>

## Exercice 11 ⭐⭐⭐ · Les jours fériés qui tombent un week-end

Un jour férié qui tombe un samedi ou un dimanche ne donne pas de jour de congé en plus. Combien y en a-t-il en France en 2026, et lesquels ?

- `feries` est redemandé dans la cellule ; convertis sa colonne `date` en vraies dates,
- `week_end` : les jours fériés dont le jour de la semaine est samedi ou dimanche (`.dt.dayofweek` vaut 5 ou 6 ; lundi vaut 0),
- `nb_week_end` et `noms_week_end` (la liste de leurs `localName`, dans l'ordre de l'année).

<details><summary>Indice</summary>

`feries["date"] = pd.to_datetime(feries["date"])`, puis `feries[feries["date"].dt.dayofweek >= 5]`.

</details>

In [ ]:
# À toi
feries = pd.DataFrame(demander("https://date.nager.at/api/v3/PublicHolidays/2026/FR", SECOURS_FERIES["FR"]))

week_end = None
nb_week_end = None
noms_week_end = None

print(nb_week_end, noms_week_end)

In [ ]:
verifier("Exercice 11", isinstance(week_end, pd.DataFrame) and nb_week_end == len(week_end) and noms_week_end == week_end.sort_values("date")["localName"].tolist() and set(pd.to_datetime(week_end["date"]).dt.dayofweek) <= {5, 6} and nb_week_end == (pd.to_datetime(feries["date"]).dt.dayofweek >= 5).sum())

<details><summary>Solution</summary>

```python
feries = pd.DataFrame(demander("https://date.nager.at/api/v3/PublicHolidays/2026/FR", SECOURS_FERIES["FR"]))

feries["date"] = pd.to_datetime(feries["date"])
week_end = feries[feries["date"].dt.dayofweek >= 5]
nb_week_end = len(week_end)
noms_week_end = week_end.sort_values("date")["localName"].tolist()

print(nb_week_end, noms_week_end)
```

Une question qu'aucun fichier ne te donne tout fait : l'API apporte la donnée brute, c'est ton code qui y trouve la réponse.

</details>

## Exercice 12 ⭐⭐⭐ · Défi : la France et l'Allemagne

Écris `feries_nationaux(pays, annee)` qui renvoie le tableau des jours fériés **nationaux** d'un pays (ceux dont la colonne `global` vaut `True` :
en Allemagne, beaucoup de jours fériés ne valent que dans certaines régions). Le secours est dans `SECOURS_FERIES[pays]`.

Puis `comparaison` : un dictionnaire `{"FR": ..., "DE": ...}` avec le nombre de jours fériés nationaux de chaque pays en 2026,
et `communs` : la liste triée des dates (en texte, `"2026-01-01"`) fériées dans les deux pays.

<details><summary>Indice</summary>

Dans la fonction : `pd.DataFrame(demander(f"https://date.nager.at/api/v3/PublicHolidays/{annee}/{pays}", SECOURS_FERIES[pays]))`, puis le filtre `t[t["global"]]`. Pour les dates communes : `sorted(set(fr["date"]) & set(de["date"]))`.

</details>

In [ ]:
# À toi
def feries_nationaux(pays, annee):
    return pd.DataFrame()

comparaison = None
communs = None

print(comparaison)
print(communs)

In [ ]:
verifier("Exercice 12", (lambda fr, de: comparaison == {"FR": len(fr), "DE": len(de)} and fr["global"].all() and de["global"].all() and communs == sorted(set(fr["date"]) & set(de["date"])) and len(de) < len(pd.DataFrame(demander("https://date.nager.at/api/v3/PublicHolidays/2026/DE", SECOURS_FERIES["DE"]))))(feries_nationaux("FR", 2026), feries_nationaux("DE", 2026)))

<details><summary>Solution</summary>

```python
def feries_nationaux(pays, annee):
    t = pd.DataFrame(demander(f"https://date.nager.at/api/v3/PublicHolidays/{annee}/{pays}", SECOURS_FERIES[pays]))
    return t[t["global"]]

fr = feries_nationaux("FR", 2026)
de = feries_nationaux("DE", 2026)
comparaison = {"FR": len(fr), "DE": len(de)}
communs = sorted(set(fr["date"]) & set(de["date"]))

print(comparaison)
print(communs)
```

Les données d'une API ne sont pas parole d'évangile : Nager.Date est un projet ouvert, tenu par des bénévoles. Pour une décision qui compte, on compare avec une source officielle.

</details>

## Bilan

La cellule ci-dessous compte tes ✅. Relance-la après chaque exercice réussi.

In [ ]:
reussis = sum(1 for ok in resultats.values() if ok)
print(f"{reussis} exercice(s) réussi(s) sur {len(resultats)}")
for nom, ok in resultats.items():
    print("  ", "✅" if ok else "❌", nom)